In [ ]:
import scipy.signal.signaltools

def _centered(arr, newsize):
    # Return the center newsize portion of the array.
    newsize = np.asarray(newsize)
    currsize = np.array(arr.shape)
    startind = (currsize - newsize) // 2
    endind = startind + newsize
    myslice = [slice(startind[k], endind[k]) for k in range(len(endind))]
    return arr[tuple(myslice)]

scipy.signal.signaltools._centered = _centered

import eventstudy as es
import numpy as np
import matplotlib.pyplot as plt

import pandas as pd
import pandas_datareader as web

In [ ]:
df = pd.read_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/dfMinute.pkl")

In [ ]:
df = df[["VIX return","Sentiment return"]]
df.dropna(inplace=True)
df = df[(df.T != 0).any()]
df = df.loc[df["Sentiment return"]!=0]
df = df.loc[df["VIX return"]!=0]
df_dates = df
df_dates["Time"] = df_dates.index
df_dates['Time']=df_dates['Time'].dt.strftime('%Y-%m-%d %H:%M:%S')
df_dates.index = df_dates['Time']

In [ ]:
event_data=df_dates["Time"]

In [ ]:
def eventstudy(returndata=df_dates, eventdata=event_data, stocklist=("VIX")):
    """
    returndata: is a dataframe with the market returns of the different firms
    eventdata: eventdata for the different firms
    stocklist: a list of the firms involved in the analysis
    
    Returns:
    abnreturn: a dictionary of the abnormal returns for each firm in their respective eventwindows -/+20
    """
    abnreturn ={}  # abnormal returns on the event window
    returndata = returndata.reset_index()
    Bse = []
    for stock in stocklist:
        
        eventindex = int(returndata[returndata['Date'] == str(eventdata.at[stock,'EventDate'])].index.values)
        print(eventindex)
        event_df = returndata.loc[eventindex-20: eventindex+20, ['Date',stock,'RF', 'Mkt_RF','SMB','HML' ]]
        estimation_df  = returndata.loc[eventindex-270: eventindex-21, ["Date",stock, "RF", "Mkt_RF","SMB","HML" ]]
        formula = stock + " - RF ~ Mkt_RF + SMB + HML"
        beta_Mkt = sm.OLS.from_formula(formula, data=estimation_df).fit().params["Mkt_RF"]
        beta_SMB = sm.OLS.from_formula(formula, data=estimation_df).fit().params["SMB"]
        beta_HML = sm.OLS.from_formula(formula, data=estimation_df).fit().params["HML"]
    
        alpha = sm.OLS.from_formula(formula, data=estimation_df).fit().params["Intercept"]
        
        standard_error = sm.OLS.from_formula(formula, data=estimation_df).fit().bse
        
        Bse.append(standard_error)
        print("{}, beta_Mkt= {}, beta_SMB = {}, beta_HML = {}, alpha= {}".format(stock, beta_Mkt, beta_SMB, beta_HML, alpha))
        
        #expected returns for each firm in the estimation window
        expectedreturn_eventwindow = ((event_df[['Mkt_RF']].values * beta_Mkt) + (event_df[['SMB']].values * 
                                       beta_SMB) + (event_df[['HML']].values * beta_HML ) + alpha)
        
        #abnormal returns on the event window - AR
       
        abnormal_return = event_df[stock].values - list(flatten(expectedreturn_eventwindow))
        abnreturn[stock] = abnormal_return
        
        
    abnormalreturns_df = pd.DataFrame(abnreturn)
    abnormalreturns_df.index = abnormalreturns_df.index-20
    return abnormalreturns_df

In [ ]:
type(eventstudy)

In [ ]:
abnormalreturns_df = pd.DataFrame(data=eventstudy(df_dates, event_data, "VIX"))
plt.figure(figsize=(24,16))
for i in range(1,17):
    plt.subplot(4,4,i)
    abnormalreturns_df[abnormalreturns_df.columns[i-1]].plot()
    plt.xlabel('Event Window')
    plt.ylabel('Return')
    plt.axhline(y=(np.sqrt((abnormalreturns_df.iloc[:,i-1].std()**2/41))*1.96),color='red',linestyle='--')
    plt.axhline(y=(np.sqrt((abnormalreturns_df.iloc[:,i-1].std()**2/41))*-1.96),color='red',linestyle='--')
    plt.title(abnormalreturns_df.columns[i-1])